In [ ]:
from common.corpus import load_corpus
from common import llm

text = {d.doc_id: d.text for d in load_corpus()}

dataset = [
    {"qid": "e1", "frage": "Wie hoch ist der zulässige Dauerbetriebsdruck?",
     "kontext": text["p12-betriebsdruck"],
     "antwort": "Der zulässige Dauerbetriebsdruck beträgt 350 bar [p12-betriebsdruck].",
     "gold": 5},
    {"qid": "e2", "frage": "Wie hoch ist der zulässige Dauerbetriebsdruck?",
     "kontext": text["p12-betriebsdruck"],
     "antwort": "Im Dauerbetrieb sind 400 bar zulässig und empfohlen.",
     "gold": 1},
    {"qid": "e3", "frage": "Welches Hydrauliköl ist vorgeschrieben?",
     "kontext": text["p12-hydraulikoel"],
     "antwort": "Vorgeschrieben ist HLP 46 nach DIN 51524 Teil 2 [p12-hydraulikoel].",
     "gold": 5},
    {"qid": "e4", "frage": "Welches Öl und in welchem Intervall wird gewechselt?",
     "kontext": text["p12-hydraulikoel"],
     "antwort": "HLP 46, Wechsel alle 1000 Betriebsstunden.",
     "gold": 2},
    {"qid": "e5", "frage": "Wann ist der erste Ölwechsel fällig?",
     "kontext": text["p12-wartungsintervalle"],
     "antwort": "Das erste Öl wird nach 50 Betriebsstunden gewechselt [p12-wartungsintervalle].",
     "gold": 5},
    {"qid": "e6", "frage": "Wie lange gilt die Werksgarantie?",
     "kontext": text["p12-datenblatt"],
     "antwort": "Die Werksgarantie beträgt 24 Monate.",
     "gold": 2},
]
print(f"{len(dataset)} Eval-Einträge, Gold-Verteilung:",
      {g: sum(1 for d in dataset if d['gold'] == g) for g in sorted({d['gold'] for d in dataset})})

In [ ]:
import json
import re

JUDGE_SYSTEM = """Sie sind ein präziser Evaluator für RAG-Antworten. Bewerten Sie
die Faithfulness (Treue zum Kontext) auf einer Skala von 1 bis 5.

Rubrik:
5 = jede Aussage wird vom Kontext gestützt.
4 = im Kern gestützt, kleine unbelegte Nebenaussage.
3 = teils gestützt, teils unbelegt.
2 = wesentliche Aussage unbelegt oder nur schwach gestützt.
1 = widerspricht dem Kontext oder ist frei erfunden.

Bewerten Sie ausschließlich anhand des Kontexts, nicht anhand Ihres Weltwissens.
Antworten Sie als JSON: {"score": <int 1-5>, "begruendung": "<ein Satz>"}."""


def judge(frage: str, kontext: str, antwort: str) -> dict:
    prompt = (f"Frage: {frage}\n\nKontext:\n{kontext}\n\nZu bewertende Antwort:\n"
              f"{antwort}\n\nIhr Urteil als JSON:")
    roh = llm.complete(prompt, system=JUDGE_SYSTEM, temperature=0.0, max_tokens=512)
    m = re.search(r"\{.*\}", roh, re.DOTALL)
    try:
        obj = json.loads(m.group(0)) if m else {}
        return {"score": int(obj.get("score", 0)), "begruendung": str(obj.get("begruendung", ""))}
    except (json.JSONDecodeError, ValueError):
        return {"score": 0, "begruendung": f"PARSE-FEHLER: {roh[:80]}"}

In [ ]:
for d in dataset:
    r = judge(d["frage"], d["kontext"], d["antwort"])
    d["judge"] = r["score"]
    print(f"{d['qid']}  gold={d['gold']}  judge={r['score']}  {r['begruendung'][:70]}")

In [ ]:
from sklearn.metrics import cohen_kappa_score

gold = [d["gold"] for d in dataset]
jud = [d["judge"] for d in dataset]
mae = sum(abs(a - b) for a, b in zip(gold, jud)) / len(gold)
try:
    kappa = cohen_kappa_score(gold, jud, weights="quadratic")
    print(f"Quadratisch gewichtetes Kappa (Mensch vs Judge): {kappa:.3f}")
except Exception as e:
    print("Kappa nicht berechenbar (zu wenig Varianz):", e)
print(f"Mittlere absolute Abweichung: {mae:.2f} Notenpunkte")
print("Faustregel: Kappa >= 0.6 ist brauchbar; darunter dem Judge nicht blind trauen.")

### Lösung

In [ ]:
grenzfall = {
    "frage": "Welcher Druck gilt und welches Öl ist vorgeschrieben?",
    "kontext": text["p12-betriebsdruck"],   # enthält Druck, aber kein Öl
    "antwort": "Der Dauerbetriebsdruck ist 350 bar und das Öl ist HLP 68.",
    "gold": 3,
}
r = judge(grenzfall["frage"], grenzfall["kontext"], grenzfall["antwort"])
print(f"gold=3  judge={r['score']}  {r['begruendung']}")
